<a href="https://colab.research.google.com/github/Trangnguyen1402/AAI2025/blob/2026Fall/customer_segmentation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# Data source:
# Telco Customer Churn Dataset
# File: WA_Fn-UseC_-Telco-Customer-Churn.csv

# Load the dataset
df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")

# Convert TotalCharges from text to numbers
df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"],
    errors="coerce"
)

# Remove rows with missing values
df = df.dropna(
    subset=[
        "tenure",
        "MonthlyCharges",
        "TotalCharges"
    ]
)

# Use 200 real customer records
df = df.sample(
    n=min(200, len(df)),
    random_state=42
).copy()

print(f"Number of customer records used: {len(df)}")

# Select customer behavior features
features = [
    "tenure",
    "MonthlyCharges",
    "TotalCharges"
]

# Scale the features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[features])

# Determine the best number of clusters using the elbow method
inertia = []
K = range(1, 6)

for k in K:
    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    kmeans.fit(X_scaled)
    inertia.append(kmeans.inertia_)

# Create and save the elbow plot
plt.figure(figsize=(8, 5))
plt.plot(K, inertia, "bo-")
plt.xlabel("Number of Clusters (K)")
plt.ylabel("Inertia")
plt.title("Elbow Method for Choosing the Number of Clusters")
plt.xticks(list(K))
plt.grid(True)
plt.savefig("elbow_plot.png")
plt.close()



# Use 3 clusters based on the elbow method
optimal_k = 3

kmeans = KMeans(
    n_clusters=optimal_k,
    random_state=42,
    n_init=10
)

# Assign each customer to a cluster
df["cluster"] = kmeans.fit_predict(X_scaled)

# Analyze the clusters
cluster_summary = (
    df.groupby("cluster")[features]
    .mean()
    .round(2)
)

print("\nCluster Characteristics:")
print(cluster_summary)

# Calculate overall averages
average_tenure = df["tenure"].mean()
average_monthly_charges = df["MonthlyCharges"].mean()
average_total_charges = df["TotalCharges"].mean()

# Recommend a strategy for each cluster
print("\nMarketing Strategies:")

for cluster in sorted(df["cluster"].unique()):
    cluster_data = cluster_summary.loc[cluster]

    print(f"\nCluster {cluster} Strategy:")

    if (
        cluster_data["TotalCharges"] > average_total_charges
        and cluster_data["tenure"] > average_tenure
    ):
        print(
            "High-value loyal customers: "
            "Offer loyalty rewards, upgrades, and exclusive benefits."
        )

    elif cluster_data["MonthlyCharges"] > average_monthly_charges:
        print(
            "High monthly-spending customers: "
            "Offer better-value plans or bundled services."
        )

    else:
        print(
            "Lower-engagement customers: "
            "Send personalized offers and re-engagement campaigns."
        )



Number of customer records used: 200

Cluster Characteristics:
         tenure  MonthlyCharges  TotalCharges
cluster                                      
0         34.95           26.19        923.07
1         14.29           74.16       1090.12
2         60.16           88.78       5310.66

Marketing Strategies:

Cluster 0 Strategy:
Lower-engagement customers: Send personalized offers and re-engagement campaigns.

Cluster 1 Strategy:
High monthly-spending customers: Offer better-value plans or bundled services.

Cluster 2 Strategy:
High-value loyal customers: Offer loyalty rewards, upgrades, and exclusive benefits.
